# 2. Phishing Link Detector Model Training
This notebook outlines how to build a Machine Learning model using TF-IDF and Logistic Regression to analyze and detect phishing URLs.

### Objective:
- Train a text-classification pipeline that extracts features from URLs.
- Evaluate model accuracy.
- Save the trained pipeline as a pickle file (`phishing_model.pkl`) to the python server directory.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

print("Scikit-Learn Imports successful.")


## 2. Dataset Generation
We construct a dataset with synthetic examples of safe and phishing URLs. Safe URLs consist of reputable domains, whereas phishing URLs contain suspicious subdomains, IP addresses, hyphens, or keywords like `kyc`, `secure-login`, `bank`, or `verify`.

In [ ]:
import os

csv_path = '../data/malicious_phish.csv'

if os.path.exists(csv_path):
    print(f"Loading custom dataset from {csv_path}...")
    df = pd.read_csv(csv_path)
    
    # Map 'type' to a binary 'label': 0 if 'benign', else 1
    df['label'] = df['type'].apply(lambda x: 0 if x.lower() == 'benign' else 1)
    
    print("Label distribution in original dataset:")
    print(df['label'].value_counts())
    
    # Subsample for efficient training (15,000 samples per class)
    # This prevents Out Of Memory errors and completes in seconds with high accuracy
    n_samples_per_class = min(15000, df['label'].value_counts().min())
    
    df_benign = df[df['label'] == 0].sample(n=n_samples_per_class, random_state=42)
    df_malicious = df[df['label'] == 1].sample(n=n_samples_per_class, random_state=42)
    df_sampled = pd.concat([df_benign, df_malicious]).sample(frac=1, random_state=42).reset_index(drop=True)
    
    X_train, X_test, y_train, y_test = train_test_split(
        df_sampled['url'], df_sampled['label'], 
        test_size=0.2, 
        random_state=42, 
        stratify=df_sampled['label']
    )
else:
    print("Custom dataset not found at default path. Using mock dataset fallback...")
    data = {
        'url': [
            'https://www.google.com', 'https://www.wikipedia.org', 'https://github.com',
            'https://www.amazon.com', 'https://www.nytimes.com', 'https://www.microsoft.com',
            'https://www.apple.com', 'https://stackoverflow.com', 'https://www.reddit.com',
            'https://www.linkedin.com', 'https://www.yahoo.com', 'https://www.medium.com',
            'http://192.168.12.33/login', 'http://sbi-kyc-verification.com/update',
            'https://secure-login-paypal.net', 'http://free-giftcard-offer.xyz',
            'http://verify-bank-details.org/kyc', 'http://h22-bank-account.support',
            'http://update-secure-access.com', 'http://win-lottery-crore.xyz',
            'http://12.44.112.50/sbi', 'http://parttime-job-deposit.net'
        ],
        'label': [
            0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
            1, 1, 1, 1, 1, 1, 1, 1, 1, 1
        ]
    }
    df = pd.DataFrame(data)
    df = pd.concat([df] * 10, ignore_index=True)
    X_train, X_test, y_train, y_test = train_test_split(
        df['url'], df['label'], 
        test_size=0.2, 
        random_state=42, 
        stratify=df['label']
    )

print(f"Data split: {len(X_train)} training, {len(X_test)} testing URLs.")

## 3. Model Pipeline & Training
We construct a vectorizer using character-level **n-grams** (3 to 5 characters) to capture structural patterns inside the URL. We then train a **Logistic Regression** model on these vectors.

In [ ]:
vectorizer = TfidfVectorizer(analyzer='char', ngram_range=(3, 5))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)

y_pred = model.predict(X_test_vec)
print("Validation Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))


## 4. Exporting the Pipeline
We bundle the fit vectorizer and model coefficients together, saving them to `phishing_model.pkl` in the `python/` directory.

In [ ]:
output_dir = '../python'
os.makedirs(output_dir, exist_ok=True)
model_path = os.path.join(output_dir, 'phishing_model.pkl')
joblib.dump({'vectorizer': vectorizer, 'model': model}, model_path)
print(f"Phishing pipeline successfully exported to {model_path}")
